# 语音识别模型训练与推理教程

本教程深入讲解语音识别 (ASR) 系统：
- 音频预处理流程
- Whisper 模型训练
- 解码策略 (Greedy, Beam Search)
- 多语言识别

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
from typing import List, Dict, Optional, Tuple

from whisper import Whisper, WhisperConfig, create_whisper_model
from audio_features import AudioFeatureExtractor, AudioConfig, create_audio_extractor

## 1. 音频预处理流程

### Whisper 预处理参数
- 采样率: 16kHz
- 窗口: 25ms (400 samples)
- 帧移: 10ms (160 samples)
- Mel 滤波器: 80

In [ ]:
class ASRPreprocessor:
    """ASR 音频预处理器"""
    
    def __init__(self, sample_rate: int = 16000, max_duration: float = 30.0):
        self.sample_rate = sample_rate
        self.max_duration = max_duration
        self.max_samples = int(sample_rate * max_duration)
        self.feature_extractor = create_audio_extractor("whisper")
    
    def pad_or_trim(self, waveform: torch.Tensor) -> torch.Tensor:
        """填充或截断到固定长度"""
        if waveform.shape[-1] > self.max_samples:
            waveform = waveform[..., :self.max_samples]
        elif waveform.shape[-1] < self.max_samples:
            pad_size = self.max_samples - waveform.shape[-1]
            waveform = F.pad(waveform, (0, pad_size))
        return waveform
    
    def process(self, waveform: torch.Tensor) -> torch.Tensor:
        """完整预处理流程"""
        # 确保单声道
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # 填充/截断
        waveform = self.pad_or_trim(waveform)
        
        # 提取 Log-Mel 特征
        mel = self.feature_extractor.extract_log_mel(waveform)
        
        return mel

# 测试
preprocessor = ASRPreprocessor()
waveform = torch.randn(16000 * 10)  # 10秒音频
mel = preprocessor.process(waveform)
print(f"Input: {waveform.shape} -> Mel: {mel.shape}")

## 2. Whisper 模型训练

In [ ]:
class WhisperTrainer:
    """Whisper 训练器"""
    
    def __init__(self, model: Whisper, lr: float = 1e-4):
        self.model = model
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        self.criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
    
    def train_step(self, mel: torch.Tensor, tokens: torch.Tensor, 
                   labels: torch.Tensor) -> float:
        """
        单步训练
        
        Args:
            mel: Log-Mel 特征 [batch, n_mels, n_frames]
            tokens: 输入 token [batch, seq_len]
            labels: 目标 token [batch, seq_len]
        """
        self.model.train()
        self.optimizer.zero_grad()
        
        # 前向传播
        logits = self.model(mel, tokens)
        
        # 计算损失
        loss = self.criterion(
            logits.view(-1, logits.size(-1)),
            labels.view(-1)
        )
        
        # 反向传播
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        
        return loss.item()

# 测试训练
model = create_whisper_model("tiny")
trainer = WhisperTrainer(model)

mel = torch.randn(2, 80, 3000)
tokens = torch.randint(0, 51865, (2, 50))
labels = torch.randint(0, 51865, (2, 50))

loss = trainer.train_step(mel, tokens, labels)
print(f"Training loss: {loss:.4f}")

## 3. 解码策略

In [ ]:
class GreedyDecoder:
    """贪婪解码器"""
    
    def __init__(self, model: Whisper):
        self.model = model
        self.config = model.config
    
    @torch.no_grad()
    def decode(self, mel: torch.Tensor, max_length: int = 224) -> torch.Tensor:
        """贪婪解码"""
        self.model.eval()
        batch_size = mel.size(0)
        device = mel.device
        
        # 编码音频
        encoder_output = self.model.encode(mel)
        
        # 初始化 token
        tokens = torch.tensor([[self.config.sot_token]], device=device)
        tokens = tokens.expand(batch_size, -1)
        
        for _ in range(max_length):
            logits = self.model.decode(tokens, encoder_output)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            tokens = torch.cat([tokens, next_token], dim=1)
            
            if (next_token == self.config.eot_token).all():
                break
        
        return tokens

class BeamSearchDecoder:
    """Beam Search 解码器"""
    
    def __init__(self, model: Whisper, beam_size: int = 5):
        self.model = model
        self.config = model.config
        self.beam_size = beam_size
    
    @torch.no_grad()
    def decode(self, mel: torch.Tensor, max_length: int = 224) -> torch.Tensor:
        """Beam Search 解码 (简化版)"""
        self.model.eval()
        device = mel.device
        
        encoder_output = self.model.encode(mel)
        
        # 初始化 beam
        tokens = torch.tensor([[self.config.sot_token]], device=device)
        scores = torch.zeros(1, device=device)
        
        for _ in range(max_length):
            # 扩展 encoder_output
            expanded_encoder = encoder_output.expand(tokens.size(0), -1, -1)
            
            logits = self.model.decode(tokens, expanded_encoder)
            log_probs = F.log_softmax(logits[:, -1, :], dim=-1)
            
            # 计算新分数
            vocab_size = log_probs.size(-1)
            new_scores = scores.unsqueeze(-1) + log_probs
            new_scores = new_scores.view(-1)
            
            # 选择 top-k
            top_scores, top_indices = new_scores.topk(self.beam_size)
            beam_indices = top_indices // vocab_size
            token_indices = top_indices % vocab_size
            
            # 更新 beam
            tokens = torch.cat([
                tokens[beam_indices],
                token_indices.unsqueeze(-1)
            ], dim=1)
            scores = top_scores
            
            if (token_indices == self.config.eot_token).all():
                break
        
        # 返回最佳序列
        return tokens[0:1]

# 测试解码器
greedy = GreedyDecoder(model)
beam = BeamSearchDecoder(model, beam_size=3)

test_mel = torch.randn(1, 80, 1500)
greedy_result = greedy.decode(test_mel, max_length=20)
print(f"Greedy decode: {greedy_result.shape}")

## 4. 多语言识别

In [ ]:
# Whisper 支持的语言 token
LANGUAGE_TOKENS = {
    "en": 50259,  # English
    "zh": 50260,  # Chinese
    "de": 50261,  # German
    "es": 50262,  # Spanish
    "ru": 50263,  # Russian
    "ko": 50264,  # Korean
    "fr": 50265,  # French
    "ja": 50266,  # Japanese
}

class MultilingualASR:
    """多语言语音识别"""
    
    def __init__(self, model: Whisper):
        self.model = model
        self.config = model.config
    
    @torch.no_grad()
    def detect_language(self, mel: torch.Tensor) -> Tuple[str, float]:
        """检测语言"""
        self.model.eval()
        
        encoder_output = self.model.encode(mel)
        
        # 使用 SOT token 开始
        tokens = torch.tensor([[self.config.sot_token]], device=mel.device)
        logits = self.model.decode(tokens, encoder_output)
        
        # 获取语言 token 的概率
        lang_probs = {}
        for lang, token_id in LANGUAGE_TOKENS.items():
            if token_id < logits.size(-1):
                lang_probs[lang] = F.softmax(logits[0, -1], dim=-1)[token_id].item()
        
        # 返回最可能的语言
        best_lang = max(lang_probs, key=lang_probs.get)
        return best_lang, lang_probs[best_lang]
    
    @torch.no_grad()
    def transcribe(self, mel: torch.Tensor, language: str = None) -> torch.Tensor:
        """转录指定语言"""
        if language is None:
            language, _ = self.detect_language(mel)
        
        # 构建初始 token 序列
        lang_token = LANGUAGE_TOKENS.get(language, LANGUAGE_TOKENS["en"])
        initial_tokens = [
            self.config.sot_token,
            lang_token,
            self.config.transcribe_token,
            self.config.no_timestamps_token
        ]
        
        return self.model.generate(mel, task="transcribe")

multilingual = MultilingualASR(model)
print(f"Supported languages: {list(LANGUAGE_TOKENS.keys())}")

## 总结

本教程介绍了语音识别的核心技术：

1. **音频预处理**: Log-Mel 特征提取
2. **模型训练**: 交叉熵损失 + 梯度裁剪
3. **解码策略**: Greedy vs Beam Search
4. **多语言**: 语言检测与指定语言转录